# Fatigue Classification with KNN — Leave-One-Task-Out (LOTO)

**Dataset:** NIOSH controlled order-picking experiment (IMU sensors + Borg CR-10 RPE).

This notebook builds **K-Nearest-Neighbours** fatigue classifiers and systematically
searches for the *input-data recipe* that gives the best results. Three things make it
different from a plain session-based model:

1. **Grouped by task, not by session name.** Each subject performed 4 sessions, and each
   session corresponds to one of 4 experimental *tasks* defined by `(Weight, Pace)`. The
   `SessionN` label is just an order index — the *same* `Session1` is a different task for
   different subjects. We map every session to its task using `experiment_settings`.

   | Task | Weight (kg) | Pace (picks/min) |
   |------|-------------|------------------|
   | 0    | 2.5         | 15               |
   | 1    | 2.5         | 10               |
   | 2    | 2.5         | 5                |
   | 3    | 1.5         | 15               |

2. **Leave-One-Task-Out (LOTO) cross-validation.** Classic LOSO holds out *all* of a
   subject's data (generalisation to a brand-new person). Here we instead hold out **one
   task at a time** across all subjects (4 folds). Every subject therefore appears in both
   train and test — on *different* tasks. This measures generalisation to a **new working
   condition for a known worker**, which is the personalised-monitoring use case. It is
   exactly this setting where subject anthropometrics are expected to help.

3. **Many input recipes + weighted variants.** We sweep sensor type (Accelerometer /
   Gyroscope / both), statistic (mean / std / rms / all), and whether subject
   anthropometrics are added. For the strongest recipes we then compare a *centralized*
   (uniform) KNN against several *weighted* KNNs (distance-weighted, correlation-weighted
   features, anthropometric emphasis).

**Excluded from all analysis:** magnetometer channels, and subjects `Sub11`, `Sub12`,
`Sub14` (per protocol / invalid data).


## 0. Imports & configuration

Everything tunable lives here so the rest of the notebook reads cleanly. Edit
`DATA_PATH` to point at your pickle.


In [ ]:
import pickle
import warnings
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH    = "NIOSH_Combined_Dataset.pkl"   # <-- point this at your pickle
OUTPUT_DIR   = Path("outputs")
CACHE_CSV    = OUTPUT_DIR / "task_features.csv"   # extracted feature matrix cache
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Cohort ───────────────────────────────────────────────────────────────────
EXCLUDE_SUBJECTS = {"Sub11", "Sub12", "Sub14"}

# ── The 4 experimental tasks (Weight kg, Pace picks/min) ─────────────────────
TASKS = [
    {"Weight": 2.5, "Pace": 15},   # task 0  (hardest: heavy + fast)
    {"Weight": 2.5, "Pace": 10},   # task 1
    {"Weight": 2.5, "Pace": 5},    # task 2
    {"Weight": 1.5, "Pace": 15},   # task 3  (lighter load)
]
TASK_NAMES = [f"T{i} (W{t['Weight']}/P{t['Pace']})" for i, t in enumerate(TASKS)]

# ── Sensors / channels (MAGNETOMETER DELIBERATELY EXCLUDED) ──────────────────
BODY_SEGMENTS = ["trunk", "upper_arm", "wrist"]
SENSOR_TYPES  = ["Accelerometer", "Gyroscope"]
AXES          = ["X", "Y", "Z"]
SENSOR_COLS   = [f"{seg}_{st}.{ax}"
                 for seg in BODY_SEGMENTS
                 for st  in SENSOR_TYPES
                 for ax  in AXES]            # 3 x 2 x 3 = 18 channels
STATS = ["mean", "std", "rms"]              # statistics computed per channel

# ── Windowing (matches the project pipeline: 10 s windows, 50% overlap) ──────
FS          = 100        # Hz
WINDOW_SIZE = 1000       # samples (10 s)
STRIDE      = 500        # samples (50% overlap)

# MVIC strength tests interrupt picking at these minutes; drop a buffer around them
MVIC_MINUTES        = [9, 18, 27, 36, 45]
MVIC_BUFFER_SECONDS = 60

# ── RPE -> 3-class fatigue label ─────────────────────────────────────────────
RPE_BIN_EDGES = [-0.01, 3.5, 6.5, 10.01]    # Low 0-3 | Moderate 4-6 | High 7-10
RPE_LABELS    = ["Low", "Moderate", "High"]

# ── Anthropometric (personalisation) features we will engineer ───────────────
ANTHRO_COLS = ["anthro_Gender", "anthro_Age", "anthro_BMI", "anthro_WHR"]

print("Config loaded. Tasks:")
for i, t in enumerate(TASKS):
    print(f"  task {i}: Weight {t['Weight']} kg, Pace {t['Pace']} picks/min")


## 1. Load & clean the dataset

`load_and_clean()` opens the pickle, drops the excluded subjects, and trims the known
recording artefact in `Sub13/Session2` (a non-monotone RPE drop at the tail). It returns
the three top-level objects: the per-session time-series, the anthropometrics table, and
the per-subject experiment-settings table.


In [ ]:
def load_and_clean(data_path):
    """
    Load the NIOSH pickle and apply the two project-standard cleaning steps.

    Returns
    -------
    ts_data    : dict[(subject, session) -> DataFrame]  IMU + RPE time-series
    anthro     : DataFrame                              one row per subject
    settings   : dict[subject -> DataFrame]             Weight/Pace per session
    """
    with open(data_path, "rb") as f:
        data = pickle.load(f)

    ts_data  = {k: v for k, v in data["ts_data"].items()
                if k[0] not in EXCLUDE_SUBJECTS}
    anthro   = data["anthro_clean"]
    anthro   = anthro[~anthro["Subject"].isin(EXCLUDE_SUBJECTS)].reset_index(drop=True)
    settings = data["experiment_settings"]

    # Trim trailing artefact RPE drop in Sub13/Session2
    key = ("Sub13", "Session2")
    if key in ts_data:
        df       = ts_data[key]
        rpe_rows = df[df["RPE_Val"].notna()]
        if not rpe_rows.empty:
            last_peak     = rpe_rows[rpe_rows["RPE_Val"] == rpe_rows["RPE_Val"].max()].index[-1]
            ts_data[key]  = df.loc[:last_peak].copy()

    n_subj = len(set(k[0] for k in ts_data))
    print(f"Loaded {len(ts_data)} sessions across {n_subj} subjects "
          f"(excluded: {sorted(EXCLUDE_SUBJECTS)})")
    return ts_data, anthro, settings


ts_data, anthro, settings = load_and_clean(DATA_PATH)
anthro.head()


## 2. Map every session to its task

`session_to_task()` looks up a subject's `experiment_settings` table and matches the
session's `(Weight, Pace)` against `TASKS`, returning the task index (0–3). This is what
lets us group by *task* instead of by the arbitrary `SessionN` order.


In [ ]:
def session_to_task(settings, subject, session, tasks=TASKS):
    """
    Resolve which task a (subject, session) belongs to.

    Matches the session's recorded Weight & Pace against the TASKS table.
    Returns the integer task index 0..len(tasks)-1, or None if no match.
    """
    if subject not in settings:
        return None

    # experiment_settings[subject] is a DataFrame indexed by 'SessionN'
    df_set = pd.DataFrame(settings[subject]).copy()
    df_set.columns = [str(c).capitalize() for c in df_set.columns]  # 'weight'->'Weight'

    if session not in df_set.index:
        return None

    weight = float(df_set.loc[session, "Weight"])
    pace   = float(df_set.loc[session, "Pace"])
    for ti, t in enumerate(tasks):
        if np.isclose(weight, t["Weight"]) and np.isclose(pace, t["Pace"]):
            return ti
    return None


# Quick audit: how many (subject, session) pairs land on each task?
_audit = {}
for (subj, sess) in ts_data:
    ti = session_to_task(settings, subj, sess)
    _audit[ti] = _audit.get(ti, 0) + 1
print("Sessions per task index:", dict(sorted(_audit.items(), key=lambda x: (x[0] is None, x[0]))))


## 3. Engineer anthropometric (personalisation) features

`build_anthro_lookup()` turns the raw anthropometrics into a compact, model-ready set per
subject:

* **Gender** → binary (M=1, F=0)
* **Age** (years)
* **BMI** = weight / height²
* **WHR** = waist / hip circumference ratio

These are constant per subject and get attached to every window from that subject. In the
LOTO setting (a known subject, a new task) they act as a personalisation signal.


In [ ]:
def build_anthro_lookup(anthro):
    """
    Build {subject -> {anthro_Gender, anthro_Age, anthro_BMI, anthro_WHR}}.

    BMI and WHR are derived; Gender is binarised. Returns a plain dict so it can be
    cheaply looked up while streaming windows.
    """
    a = anthro.copy()
    a["anthro_Gender"] = a["Gender"].astype(str).str.upper().str[0].map({"M": 1, "F": 0})
    a["anthro_Age"]    = pd.to_numeric(a["Age"], errors="coerce")
    a["anthro_BMI"]    = a["Weight (kg)"] / (a["Height (cm)"] / 100.0) ** 2
    a["anthro_WHR"]    = a["Waist circumference (cm)"] / a["Hip circumference (cm)"]
    return a.set_index("Subject")[ANTHRO_COLS].to_dict("index")


anthro_lkp = build_anthro_lookup(anthro)
pd.DataFrame(anthro_lkp).T.head()


## 4. Windowed feature extraction

For each session we slide a 10 s window (50% overlap), skip windows that fall inside an
MVIC strength-test buffer or contain NaNs, and compute **mean / std / rms** for each of
the 18 accelerometer+gyroscope channels (54 sensor features). The label for a window is
the median interpolated RPE inside it, binned to Low/Moderate/High. Anthropometrics and
the task index are attached to every row.

The result is cached to CSV so you only pay the extraction cost once.


In [ ]:
def interpolate_rpe(df):
    """Linearly interpolate the sparse RPE column across all rows of one session."""
    rpe = df["RPE_Val"].astype(float).interpolate(method="linear")
    return rpe.bfill().ffill()


def bin_rpe(value):
    """Map a scalar RPE to class index 0=Low, 1=Moderate, 2=High."""
    out = pd.cut([value], bins=RPE_BIN_EDGES, labels=[0, 1, 2])[0]
    if pd.isna(out):
        return 0 if value <= 3.5 else (1 if value <= 6.5 else 2)
    return int(out)


def mvic_exclusion_mask(df):
    """True where a row is within MVIC_BUFFER_SECONDS of any MVIC strength test."""
    elapsed = df["Timestamp"].values - df["Timestamp"].iloc[0]
    mask = np.zeros(len(df), dtype=bool)
    for t_min in MVIC_MINUTES:
        mask |= np.abs(elapsed - t_min * 60) <= MVIC_BUFFER_SECONDS
    return mask


def window_stats(window, channels):
    """
    Compute mean / std / rms per channel for one (window_size, n_channels) array.
    Returns a flat {f'{channel}_{stat}': value} dict.
    """
    mean = window.mean(axis=0)
    std  = window.std(axis=0)
    rms  = np.sqrt((window ** 2).mean(axis=0))
    feats = {}
    for j, ch in enumerate(channels):
        feats[f"{ch}_mean"] = float(mean[j])
        feats[f"{ch}_std"]  = float(std[j])
        feats[f"{ch}_rms"]  = float(rms[j])
    return feats


def build_feature_matrix(ts_data, settings, anthro_lkp, cache_csv=CACHE_CSV):
    """
    Slide windows over every session and assemble one labelled feature row per window.

    Each row carries: Subject, Task (int) / TaskName, y (0/1/2),
    54 sensor features (18 channels x mean/std/rms) and the 4 anthropometric features.

    Uses the CSV cache if present. Returns the full feature DataFrame.
    """
    if Path(cache_csv).exists():
        print(f"Loading cached feature matrix from {cache_csv} ...")
        return pd.read_csv(cache_csv)

    # canonical channel order = the acc+gyro columns that exist in the data
    channels = [c for c in SENSOR_COLS]
    rows = []

    for (subj, sess), df in sorted(ts_data.items()):
        task = session_to_task(settings, subj, sess)
        if task is None:
            continue

        # keep only this session's rows if a Session column is present
        if "Session" in df.columns:
            df = df[df["Session"].astype(str).str.strip() == sess].copy()
        if df.empty:
            continue

        avail = [c for c in channels if c in df.columns]
        if len(avail) != len(channels):     # need all 18 channels for a fair comparison
            print(f"  skip {subj}/{sess}: missing channels")
            continue

        rpe   = interpolate_rpe(df).values
        imu   = df[channels].values.astype(np.float32)
        mvic  = mvic_exclusion_mask(df)
        anth  = anthro_lkp.get(subj, {})
        n     = len(df)

        n_win, start = 0, 0
        while start + WINDOW_SIZE <= n:
            end = start + WINDOW_SIZE
            sl  = slice(start, end)

            if mvic[sl].any() or np.isnan(imu[sl]).any():
                start += STRIDE
                continue

            rpe_slice = rpe[sl]
            rpe_slice = rpe_slice[~np.isnan(rpe_slice)]
            if rpe_slice.size == 0:
                start += STRIDE
                continue

            row = {"Subject": subj, "Task": task, "TaskName": TASK_NAMES[task],
                   "y": bin_rpe(float(np.median(rpe_slice)))}
            row.update(window_stats(imu[sl], channels))
            for k in ANTHRO_COLS:
                row[k] = anth.get(k, np.nan)
            rows.append(row)
            n_win += 1
            start += STRIDE

        print(f"  {subj}/{sess} -> task {task}: {n_win} windows")

    feat_df = pd.DataFrame(rows)
    feat_df.to_csv(cache_csv, index=False)
    print(f"\nFeature matrix: {len(feat_df):,} windows x {feat_df.shape[1]} cols "
          f"-> cached to {cache_csv}")
    return feat_df


feat_df = build_feature_matrix(ts_data, settings, anthro_lkp)

# Drop any rows with missing features, report class & task balance
SENSOR_FEATS = [f"{ch}_{s}" for ch in SENSOR_COLS for s in STATS]
feat_df = feat_df.dropna(subset=SENSOR_FEATS + ANTHRO_COLS + ["y"]).reset_index(drop=True)

print("\nClass balance:")
print(feat_df["y"].map(dict(enumerate(RPE_LABELS))).value_counts())
print("\nWindows per task:")
print(feat_df["TaskName"].value_counts().sort_index())


## 5. Input-recipe selection + model factory

Two ingredients drive the comparison:

**`select_columns(...)`** returns the feature columns for a given recipe — a choice of
sensor types, statistics, and whether anthropometrics are appended.

**Model modes** (`make_model(...)`):

* `centralized` — plain KNN, **uniform** vote (every neighbour counts equally).
* `distance` — KNN with **distance-weighted** voting (closer neighbours count more).
* `corr_weighted` — features are scaled by their |correlation with the label| *before*
  the KNN distance is computed, so informative features dominate the metric. Fitted **only
  on training data inside each fold** (no leakage), then distance-weighted KNN.
* `anthro_boost` — anthropometric features are multiplied by a boost factor so the metric
  leans on personalisation, then distance-weighted KNN.

`CorrelationWeighter` and `BlockWeighter` are small sklearn transformers, so they live
inside a `Pipeline` and are re-fit per fold automatically.


In [ ]:
def select_columns(sensors=("Accelerometer", "Gyroscope"),
                   stats=("mean", "std", "rms"),
                   anthro=False):
    """
    Build the list of feature columns for one input recipe.

    sensors : which sensor types to include (subset of SENSOR_TYPES)
    stats   : which per-channel statistics to include (subset of STATS)
    anthro  : append the 4 anthropometric personalisation features?
    """
    cols = [f"{ch}_{st}"
            for ch in SENSOR_COLS
            for st in stats
            if any(s in ch for s in sensors)]
    if anthro:
        cols = cols + ANTHRO_COLS
    return cols


class CorrelationWeighter(BaseEstimator, TransformerMixin):
    """
    Supervised feature weighter: multiplies each (already-scaled) feature by the
    absolute Pearson correlation between that feature and the (ordinal) label.
    Features that track fatigue get a longer reach in the KNN distance.
    Fit on TRAIN ONLY (enforced by living inside the Pipeline).
    """
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        w = np.zeros(X.shape[1])
        for j in range(X.shape[1]):
            col = X[:, j]
            if np.std(col) > 1e-12:
                c = np.corrcoef(col, y)[0, 1]
                w[j] = abs(c) if np.isfinite(c) else 0.0
        if w.sum() == 0:          # degenerate guard
            w[:] = 1.0
        self.w_ = w
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float) * self.w_


class BlockWeighter(BaseEstimator, TransformerMixin):
    """
    Multiplies a fixed subset of columns (boolean mask) by `factor`. Used to emphasise
    the anthropometric block so the metric leans on personalisation.
    """
    def __init__(self, mask, factor=2.0):
        self.mask = np.asarray(mask, dtype=bool)
        self.factor = factor

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        X[:, self.mask] *= self.factor
        return X


def make_model(mode, k, metric="minkowski", anthro_mask=None):
    """
    Build a KNN pipeline for one of the four modes.
    StandardScaler is always first so every feature starts on equal footing,
    and (critically) it is re-fit per CV fold.
    """
    scaler = ("scaler", StandardScaler())
    if mode == "centralized":
        return Pipeline([scaler,
                         ("knn", KNeighborsClassifier(n_neighbors=k, weights="uniform",
                                                      metric=metric, n_jobs=-1))])
    if mode == "distance":
        return Pipeline([scaler,
                         ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance",
                                                      metric=metric, n_jobs=-1))])
    if mode == "corr_weighted":
        return Pipeline([scaler, ("cw", CorrelationWeighter()),
                         ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance",
                                                      metric=metric, n_jobs=-1))])
    if mode == "anthro_boost":
        if anthro_mask is None or not anthro_mask.any():
            raise ValueError("anthro_boost requires an anthro_mask with anthropometric cols")
        return Pipeline([scaler, ("bw", BlockWeighter(anthro_mask, factor=2.0)),
                         ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance",
                                                      metric=metric, n_jobs=-1))])
    raise ValueError(f"unknown mode {mode!r}")


## 6. Hyperparameter tuning + Leave-One-Task-Out evaluation

`tune_k()` picks the neighbour count `k` with a **subject-grouped** inner CV so no
subject straddles the train/validation split during tuning. We then report performance
with `loto_cv()`, which holds out **one task at a time** (4 folds). Inside every LOTO fold
the whole pipeline — scaler, any weighter, and KNN — is fit on the 3 training tasks only,
giving an unbiased estimate.


In [ ]:
K_GRID = [3, 5, 7, 9, 11, 15, 21]


def tune_k(X, y, groups, mode, anthro_mask=None, k_grid=K_GRID):
    """
    Choose k via subject-grouped GridSearch (scoring = macro-F1).
    Grouping by subject prevents the same person appearing on both sides of a split.
    """
    n_groups = len(np.unique(groups))
    cv = GroupKFold(n_splits=min(5, n_groups))
    base = make_model(mode, k=k_grid[0], anthro_mask=anthro_mask)
    search = GridSearchCV(base, {"knn__n_neighbors": k_grid},
                          scoring="f1_macro", cv=cv, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_params_["knn__n_neighbors"]


def loto_cv(feat_df, cols, mode, k=None, return_preds=False):
    """
    Leave-One-Task-Out cross-validation.

    For each task t (0..3): train on the other 3 tasks, test on task t. The pipeline is
    re-fit per fold so the scaler/weighter never see the held-out task.

    If k is None it is tuned (subject-grouped) on the full data once, then reused across
    folds — k is a coarse knob, while LOTO provides the unbiased performance estimate.

    Returns a results dict (macro-F1, accuracy, per-task F1, k) and, optionally, the
    stacked y_true / y_pred for plotting.
    """
    X    = feat_df[cols].values.astype(float)
    y    = feat_df["y"].values.astype(int)
    task = feat_df["Task"].values.astype(int)
    subj = feat_df["Subject"].values

    anthro_mask = np.array([c in ANTHRO_COLS for c in cols])

    if k is None:
        k = tune_k(X, y, groups=subj, mode=mode, anthro_mask=anthro_mask)

    y_true_all, y_pred_all, fold_task = [], [], []
    per_task_f1 = {}
    for t in sorted(np.unique(task)):
        tr, te = task != t, task == t
        model = make_model(mode, k=k, anthro_mask=anthro_mask)
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])
        per_task_f1[int(t)] = f1_score(y[te], pred, average="macro", zero_division=0)
        y_true_all.extend(y[te]); y_pred_all.extend(pred); fold_task.extend([t] * te.sum())

    y_true_all = np.array(y_true_all); y_pred_all = np.array(y_pred_all)
    res = {
        "mode": mode, "k": int(k), "n_features": len(cols),
        "macro_f1": f1_score(y_true_all, y_pred_all, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
    }
    for t, v in per_task_f1.items():
        res[f"f1_task{t}"] = v
    if return_preds:
        return res, y_true_all, y_pred_all, np.array(fold_task)
    return res


## 7. Stage 1 — which *input recipe* works best?

We sweep every combination of **sensor type × statistic × anthropometrics** and evaluate
each under the distance-weighted KNN (a strong, fast baseline). The table is sorted by
LOTO macro-F1 so the best input recipe rises to the top.


In [ ]:
# Build the recipe grid
SENSOR_SETS = {
    "acc":      ("Accelerometer",),
    "gyro":     ("Gyroscope",),
    "acc+gyro": ("Accelerometer", "Gyroscope"),
}
STAT_SETS = {
    "mean": ("mean",),
    "std":  ("std",),
    "rms":  ("rms",),
    "all":  ("mean", "std", "rms"),
}

stage1_rows = []
for (sname, sensors), (stname, stats), anthro in itertools.product(
        SENSOR_SETS.items(), STAT_SETS.items(), [False, True]):
    cols = select_columns(sensors=sensors, stats=stats, anthro=anthro)
    res  = loto_cv(feat_df, cols, mode="distance")
    res["sensors"] = sname
    res["stats"]   = stname
    res["anthro"]  = anthro
    res["recipe"]  = f"{sname}|{stname}|{'+anthro' if anthro else 'noanthro'}"
    stage1_rows.append(res)
    print(f"{res['recipe']:32s}  k={res['k']:2d}  macroF1={res['macro_f1']:.3f}  acc={res['accuracy']:.3f}")

stage1 = pd.DataFrame(stage1_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
cols_show = ["recipe", "sensors", "stats", "anthro", "k", "n_features", "macro_f1", "accuracy"]
stage1[cols_show].head(15)


### Does adding anthropometrics help?

Paired comparison of each sensor/stat recipe with vs without the personalisation block.


In [ ]:
piv = stage1.pivot_table(index=["sensors", "stats"], columns="anthro",
                         values="macro_f1")
piv.columns = ["no_anthro", "with_anthro"]
piv["delta"] = piv["with_anthro"] - piv["no_anthro"]
piv = piv.sort_values("delta", ascending=False)
print("Macro-F1 change from adding anthropometrics (positive = helps):")
piv.round(4)


## 8. Stage 2 — centralized vs weighted models on the best recipes

Take the top recipes from Stage 1 and compare all four KNN modes. `anthro_boost` only
runs where the recipe actually contains anthropometrics.


In [ ]:
TOP_N = 4
top_recipes = stage1.head(TOP_N)[["recipe", "sensors", "stats", "anthro"]].to_dict("records")

stage2_rows = []
for r in top_recipes:
    sensors = SENSOR_SETS[r["sensors"]]
    stats   = STAT_SETS[r["stats"]]
    cols    = select_columns(sensors=sensors, stats=stats, anthro=r["anthro"])

    modes = ["centralized", "distance", "corr_weighted"]
    if r["anthro"]:
        modes.append("anthro_boost")

    for mode in modes:
        res = loto_cv(feat_df, cols, mode=mode)
        res["recipe"] = r["recipe"]
        stage2_rows.append(res)
        print(f"{r['recipe']:32s} {mode:14s}  k={res['k']:2d}  "
              f"macroF1={res['macro_f1']:.3f}  acc={res['accuracy']:.3f}")

stage2 = pd.DataFrame(stage2_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
stage2[["recipe", "mode", "k", "macro_f1", "accuracy",
        "f1_task0", "f1_task1", "f1_task2", "f1_task3"]].head(20)


## 9. Visual comparison & best-model diagnostics

A grouped bar chart of mode performance on the top recipes, plus the confusion matrix and
per-task macro-F1 for the single best (recipe, mode) combination.


In [ ]:
# (a) grouped bars: mode comparison across top recipes
fig, ax = plt.subplots(figsize=(11, 5))
modes_order = ["centralized", "distance", "corr_weighted", "anthro_boost"]
mode_colors = {"centralized": "#457B9D", "distance": "#1D3557",
               "corr_weighted": "#E63946", "anthro_boost": "#2A9D8F"}
recipes = stage2["recipe"].unique().tolist()
x = np.arange(len(recipes)); w = 0.2
for i, mode in enumerate(modes_order):
    vals = [stage2[(stage2.recipe == rc) & (stage2["mode"] == mode)]["macro_f1"].max()
            for rc in recipes]
    vals = [0 if pd.isna(v) else v for v in vals]
    ax.bar(x + (i - 1.5) * w, vals, w, label=mode, color=mode_colors[mode])
ax.set_xticks(x); ax.set_xticklabels(recipes, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("LOTO macro-F1"); ax.set_ylim(0, 1)
ax.set_title("Centralized vs weighted KNN across top input recipes", fontweight="bold")
ax.legend(title="mode", fontsize=9); ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "mode_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# (b) refit the overall winner and show its confusion matrix + per-task F1
best = stage2.iloc[0]
best_r = next(r for r in top_recipes if r["recipe"] == best["recipe"])
best_cols = select_columns(sensors=SENSOR_SETS[best_r["sensors"]],
                           stats=STAT_SETS[best_r["stats"]],
                           anthro=best_r["anthro"])
res, y_true, y_pred, fold_task = loto_cv(feat_df, best_cols, mode=best["mode"],
                                         return_preds=True)
print(f"BEST MODEL  ->  recipe={best['recipe']}  mode={best['mode']}  "
      f"k={res['k']}  macroF1={res['macro_f1']:.3f}  acc={res['accuracy']:.3f}\n")
print(classification_report(y_true, y_pred, target_names=RPE_LABELS, zero_division=0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=RPE_LABELS).plot(ax=ax1, cmap="Blues", colorbar=False)
ax1.set_title(f"Confusion matrix\n{best['recipe']} | {best['mode']}")

task_f1 = [f1_score(y_true[fold_task == t], y_pred[fold_task == t],
                    average="macro", zero_division=0) for t in range(len(TASKS))]
bars = ax2.bar(TASK_NAMES, task_f1, color="#1D3557")
ax2.axhline(res["macro_f1"], color="#E63946", ls="--",
            label=f"overall = {res['macro_f1']:.3f}")
ax2.set_ylim(0, 1); ax2.set_ylabel("macro-F1 (held-out task)")
ax2.set_title("Per-task LOTO performance"); ax2.legend()
plt.setp(ax2.get_xticklabels(), rotation=20, ha="right", fontsize=9)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "best_model_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Conclusions

This cell prints a programmatic summary: the best input recipe, whether anthropometrics
helped on average, and whether a weighted model beat the centralized baseline. All result
tables and figures are saved under `outputs/`.


In [ ]:
# Save full result tables
stage1.to_csv(OUTPUT_DIR / "stage1_input_recipes.csv", index=False)
stage2.to_csv(OUTPUT_DIR / "stage2_model_modes.csv", index=False)

best_input   = stage1.iloc[0]
anthro_gain  = piv["delta"].mean()
base_f1      = stage2[stage2["mode"] == "centralized"]["macro_f1"].max()
best_f1      = stage2["macro_f1"].max()
best_overall = stage2.iloc[0]

print("=" * 70)
print("SUMMARY  (Leave-One-Task-Out evaluation)")
print("=" * 70)
print(f"Best INPUT recipe (Stage 1) : {best_input['recipe']}")
print(f"    sensors={best_input['sensors']}, stats={best_input['stats']}, "
      f"anthro={best_input['anthro']}, k={best_input['k']}")
print(f"    macro-F1={best_input['macro_f1']:.3f}, accuracy={best_input['accuracy']:.3f}")
print(f"\nAnthropometrics: average macro-F1 change when added = {anthro_gain:+.4f} "
      f"({'helps' if anthro_gain > 0 else 'hurts/neutral'} on average)")
print(f"\nBest OVERALL model (Stage 2): {best_overall['recipe']} | {best_overall['mode']}")
print(f"    macro-F1={best_overall['macro_f1']:.3f}, accuracy={best_overall['accuracy']:.3f}")
print(f"    centralized baseline macro-F1 = {base_f1:.3f} -> "
      f"weighted gain = {best_f1 - base_f1:+.4f}")
print("\nSaved: stage1_input_recipes.csv, stage2_model_modes.csv, "
      "mode_comparison.png, best_model_diagnostics.png  (in outputs/)")
print("=" * 70)
